In [ ]:
import json
import csv
from pathlib import Path

def compare_and_export_matches(base_csv_directory, json_file_path, output_csv_path, csv_column="repo_name"):
    """
    Compares repos and exports overlaps to a CSV, tracking Stars, Language, Algorithm, and Tier.
    """
    csv_dir = Path(base_csv_directory)
    json_path = Path(json_file_path)
    
    # Potential column names for star counts in CSV files
    star_col_names = ["stars", "stargazers_count", "star_count", "stars_count"]
    
    # Stores repo name -> list of dicts with file path and star count
    my_csv_dataset = {}
    
    csv_files = list(csv_dir.glob("**/*.csv"))
    
    if not csv_files:
        print(f"Error: No CSV files found in '{csv_dir.resolve()}'")
        return

    print(f" Found {len(csv_files)} CSV files. Reading CSV data...")

    for file in csv_files:
        try:
            rel_path = str(file.relative_to(csv_dir))
            
            with open(file, 'r', encoding='utf-8') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    if csv_column in row and row[csv_column].strip():
                        repo_name = row[csv_column].strip().lower()
                        
                        # Find star count from row if available
                        star_count = "N/A"
                        for col in star_col_names:
                            if col in row and row[col].strip():
                                star_count = row[col].strip()
                                break
                        
                        if repo_name not in my_csv_dataset:
                            my_csv_dataset[repo_name] = []
                        
                        entry = {"path": rel_path, "stars": star_count}
                        if entry not in my_csv_dataset[repo_name]:
                            my_csv_dataset[repo_name].append(entry)
        except Exception as e:
            print(f"Warning: Could not read {file.name} - {e}")

    print("Reading Joe's JSON file...")
    try:
        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            raw_items = data.get("items", [])
            his_json_dataset = set(item.strip().lower() for item in raw_items if isinstance(item, str))
    except Exception as e:
        print(f"Error reading JSON: {e}")
        return
    # Find overlapping repositories
    common_repos = set(my_csv_dataset.keys()).intersection(his_json_dataset)
    
    print(f"\n Found {len(common_repos):,} overlapping repositories.")
    print(f"Saving results to:\n   {output_csv_path}")
    
    # Write output CSV
    with open(output_csv_path, 'w', newline='', encoding='utf-8') as out_f:
        writer = csv.writer(out_f)
        writer.writerow(["Repository", "Stars", "Language", "Algorithm", "Tier File", "Full Relative Path"])
        
        for repo in sorted(common_repos):
            for item in my_csv_dataset[repo]:
                path_str = item["path"]
                stars = item["stars"]
                parts = Path(path_str).parts
                
                if len(parts) >= 3:
                    language = parts[0]
                    algorithm = parts[1]
                    tier = parts[-1]
                else:
                    language, algorithm, tier = "Unknown", "Unknown", "Unknown"
                    
                writer.writerow([repo, stars, language, algorithm, tier, path_str])
                
    print("Export complete")

#  Execution
if __name__ == "__main__":
    csv_folder = "yourcsvfolder"  # Change this to your source folder
    json_file = "yourjsonfile.json"  # Change this to your JSON file path
    output_file = "youroutputfile.csv"  # Change this to your desired output file path

    compare_and_export_matches(
        base_csv_directory=csv_folder, 
        json_file_path=json_file, 
        output_csv_path=output_file,
        csv_column="repo_name"  # Update if your repo column name is different
    )